# In our existing project Suppose we got a requirement to add more Documents to our existing persisted Vector Database(RAG_PDF_Analysis_System.ipynb)

## loading env to notebook & checking OPENAI_API is connected and working fine

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['OPENAI_API_KEY']:
    print("key is set")


key is set


# STEP 0: CREATE LLM with ChatOpenAI function with the model="gpt-5-nano"

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# STEP 1: extrating text from PDF using Langchain


In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = '../Docs/Animal_Planet_Animal_Lifecycle.pdf'

loader = PyPDFLoader(pdf_path)

docs = loader.load()
docs

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-11T07:53:28+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-11T07:53:28+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../Docs/Animal_Planet_Animal_Lifecycle.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Animal Planet\nChapter 1: Birth and Early Life\nThe lifecycle of animals begins with birth or\nhatching. Mammals such as lions and elephants\ngive birth to live young, while birds, reptiles, and\namphibians hatch from eggs. The early stage of\nlife is critical because baby animals depend\nheavily on protection, warmth, nutrition, and\nparental care for survival. Young animals spend\nmuch of their time learning survival skills. Lion\ncubs observe hunting techniques from adults, baby\nelephants stay\nclose to their mothers, and birds learn flying skills\nbefore leaving the

# STEP 2: CREATING CHUNKS OF IT

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100, 
    chunk_overlap=5
    )

chunks = splitter.split_documents(docs)
len(chunks)
chunks[0].metadata

{'producer': 'ReportLab PDF Library - www.reportlab.com',
 'creator': '(unspecified)',
 'creationdate': '2026-05-11T07:53:28+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-05-11T07:53:28+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': '../Docs/Animal_Planet_Animal_Lifecycle.pdf',
 'total_pages': 4,
 'page': 0,
 'page_label': '1'}

# STEP 3: CREATE EMBEDDINGS Model variable


In [11]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# STEP 3 :STORE EMBEDDINGS IN THE EXISTING LOCAL VECTOR STORE

In [14]:
from langchain_community.vectorstores import Chroma

#establishing connection with existing vector DB
vectorstore = Chroma(
    persist_directory = "../Vector/",
    embedding_function=embedding_model
    )

#adding my chunks to connected Vector DB
vectorstore.add_documents(chunks)


['1b7c13c8-72c0-4237-9000-739ebc7a5b1a',
 '37a3b282-f95b-40a2-804e-a5ae9e1e1438',
 'fd8f6de5-e49f-4cba-ba92-e7091dcbcf3f',
 'c9165b18-85c7-419d-97f0-20fefe24b878',
 'ee963c0e-94dd-4109-b31e-83451faafec6',
 'a6713a30-af8c-4812-9b82-b043b899dd4a',
 '6fcc0cdf-8178-4e4c-aeeb-df29336082e9',
 '0dacdce7-8e74-4c8d-af3d-0e2f4aa96fdf',
 '20b5da88-283f-42ee-a058-2282861683bb',
 'a1c9b0f7-623b-4a2d-bb09-4333a2d47c52',
 '379bf441-a651-437a-a824-de3e8c079c37',
 '3cb07cad-593f-4005-b18d-c61bce843128',
 '6fd3817f-840b-492c-baa6-2f6a57fc87c8',
 '8b9acac9-51a8-4ea0-8f42-ae4a75fd9dcf',
 '487de9a6-34a9-4ca1-a126-f9d5d18c24e9',
 'a1e28817-7f99-47e3-8a3f-0bf7478096ad',
 '971c8682-01f7-44ee-8da4-185d9f8e45f3',
 '312b4074-de57-4d14-b6d2-7970d58b58f1',
 'bb300afa-0bdf-4fa3-9b18-648920d9a1c0',
 '872c9968-0956-49a3-a324-9e69761baadf',
 '9973cc52-41ec-44d2-841a-ef17fa6ffaa0',
 'ea3d7572-f2c9-4b58-8a40-77da897e9fd8',
 '410b13a7-4af8-4c1f-aa22-1af5ceb0008a',
 '5f4b7554-05b0-49c5-bc52-32428bcf5604',
 'a6d10c78-c1d5-

# STEP 5: TESTING : SIMILARITY SEARCH

In [15]:
ans = vectorstore.similarity_search("What is THE Life cycle of Dolphin", k=3)
ans

[Document(metadata={'trapped': '/False', 'creator': '(unspecified)', 'source': '../Docs/Animal_Planet_Detailed_Book.pdf', 'keywords': '', 'page': 4, 'producer': 'ReportLab PDF Library - www.reportlab.com', 'total_pages': 11, 'subject': '(unspecified)', 'creationdate': '2026-05-11T04:06:04+00:00', 'author': '(anonymous)', 'page_label': '5', 'title': '(anonymous)', 'moddate': '2026-05-11T04:06:04+00:00'}, page_content='Chapter 4: Dolphin - Masters of the Ocean\nDolphins are intelligent marine mammals famous for'),
 Document(metadata={'title': '(anonymous)', 'keywords': '', 'trapped': '/False', 'creationdate': '2026-05-11T04:06:04+00:00', 'author': '(anonymous)', 'creator': '(unspecified)', 'page': 9, 'producer': 'ReportLab PDF Library - www.reportlab.com', 'source': '../Docs/Animal_Planet_Detailed_Book.pdf', 'moddate': '2026-05-11T04:06:04+00:00', 'total_pages': 11, 'subject': '(unspecified)', 'page_label': '10'}, page_content='Chapter 9: Dolphin - Masters of the Ocean\nDolphins are inte